# Jurisight – Part 1: Data Handling (Colab/Drive)

This notebook focuses on loading legal case data from Google Drive, cleaning it, and preparing it for model training.

**Datasets referenced in the project documents:**
- ECHR (European Court of Human Rights)
- ILDC (Indian Legal Documents Corpus)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install dependencies

Adjust versions if your project pins specific packages.


In [ ]:
!pip -q install pandas numpy datasets scikit-learn matplotlib seaborn

## Configure data paths

Place your datasets in Google Drive, for example:
`/content/drive/MyDrive/Jurisight/data/`


In [ ]:
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Jurisight/data')
ECHR_PATH = DATA_ROOT / 'echr.csv'
ILDC_PATH = DATA_ROOT / 'ildc.csv'
DATA_ROOT

## Load data

Update columns to match your dataset schema. Expected columns for training:
- `text`: concatenated legal document text
- `label`: verdict label (e.g., 0/1 or multi-class)


In [ ]:
import pandas as pd

def load_dataset(path):
    if not path.exists():
        raise FileNotFoundError(f'Expected dataset at {path}')
    df = pd.read_csv(path)
    return df

echr_df = load_dataset(ECHR_PATH)
echr_df.head()

## Basic cleaning & inspection

Customize this section based on the data quality and structure of your corpus.


In [ ]:
def basic_cleaning(df):
    df = df.dropna(subset=['text', 'label']).copy()
    df['text'] = df['text'].astype(str).str.replace('\s+', ' ', regex=True).str.strip()
    return df

echr_df = basic_cleaning(echr_df)
echr_df['label'].value_counts().head()

## Train/validation/test split

Persist splits back to Drive for reuse in training.


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(echr_df, test_size=0.2, random_state=42, stratify=echr_df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

splits_dir = DATA_ROOT / 'splits'
splits_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(splits_dir / 'train.csv', index=False)
val_df.to_csv(splits_dir / 'val.csv', index=False)
test_df.to_csv(splits_dir / 'test.csv', index=False)
splits_dir